# F-POD 风格点击信号多途聚类分析

本 Notebook 将原 `main.py` 的完整流程拆分为多个可独立检查的步骤。  
每个代码单元都会输出总体进度和当前阶段的关键结果，便于逐步运行、检查和调试。

处理流程：环境与模块配置 → 路径设置 → WAV 读取 → 带通滤波 → F-POD 流程 → CSV 导出 → 聚类绘图 → 结果汇总。

## 第 1 步：配置运行环境并导入模块

先确定项目与模块目录，再配置 Matplotlib，最后导入信号处理、F-POD 分析和可视化函数。

In [1]:
# 导入操作系统模块，用于配置 Matplotlib 的运行环境变量。
import os
# 导入系统模块，用于把本项目的模块目录加入 Python 搜索路径。
import sys
# 导入 Path 类，以跨平台方式构造、检查和解析文件路径。
from pathlib import Path

# 指定当前 Notebook 所在的项目功能目录，后续从该目录加载 src 包。
MODULE_DIR = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin\16_FpodStyleMultipath")
# 如果模块目录尚未加入 Python 搜索路径，则准备将其插入搜索路径首位。
if str(MODULE_DIR) not in sys.path:
    # 将模块目录放在搜索路径首位，确保导入的是当前项目中的 src 包。
    sys.path.insert(0, str(MODULE_DIR))

# 在导入可视化模块前指定 Matplotlib 配置目录，避免配置文件写入用户目录。
os.environ.setdefault("MPLCONFIGDIR", str(MODULE_DIR / ".matplotlib"))

# 导入点击检测配置、带通滤波函数和单通道 WAV 读取函数。
from src.click_detection import DetectionSettings, bandpass_filter, read_wav_mono
# 导入 F-POD 流程配置、主处理函数和点击记录 CSV 写出函数。
from src.fpod_pdf_pipeline import FpodPdfSettings, run_fpod_pdf_pipeline, write_fpod_records_csv
# 导入多途聚类绘图函数和特征 CSV 写出函数。
from src.visualization import plot_multipath_cluster, write_features_csv
# 导入 Notebook 图像显示工具，用于在单元格中直接预览生成的 PNG。
from IPython.display import Image, display

# 输出总体进度，说明运行环境与项目模块已经准备完成。
print("[###.............................]  10% 运行环境与模块导入完成")

[###.............................]  10% 运行环境与模块导入完成


## 第 2 步：设置输入与输出路径

如需分析其他录音，只需修改本单元中的 `WAV_PATH`。所有输出文件集中保存在 `OUTPUT_DIR`。

In [2]:
# 指定项目根目录，作为输入数据和分析结果路径的共同基准。
BASE_DIR = Path(r"D:\Project_Github\Indo-Pacific-humpback-dolphin")
# 指定待分析的脉冲序列 WAV 文件；分析其他文件时可修改此变量。
WAV_PATH = BASE_DIR / "00_Data" / "ClickTrains" / "PulseTrain_651" / "PulseTrain.wav"
# 指定所有 CSV 和 PNG 分析结果的保存目录。
OUTPUT_DIR = BASE_DIR / "16_FpodStyleMultipath"

# 将输入文件路径解析为规范的绝对路径，避免相对工作目录造成歧义。
wav_path = WAV_PATH.resolve()
# 将输出目录解析为规范的绝对路径。
output_dir = OUTPUT_DIR.resolve()
# 定义完整时间范围聚类图的输出路径。
png_path = output_dir / "multipath_cluster_visualization.png"
# 定义 80—120 毫秒局部聚类图的输出路径。
cropped_png_path = output_dir / "multipath_cluster_visualization_80_120ms.png"
# 定义 30—130 毫秒局部聚类图的输出路径。
cropped_30_130_png_path = output_dir / "multipath_cluster_visualization_30_130ms.png"
# 定义多途特征表的 CSV 输出路径。
csv_path = output_dir / "multipath_features.csv"
# 定义类似 FP1 的底层点击记录 CSV 输出路径。
records_csv_path = output_dir / "fpod_click_records.csv"

# 检查输入 WAV 文件是否存在，以便在耗时处理前发现路径错误。
if not wav_path.exists():
    # 输入文件缺失时抛出包含完整路径的异常，并停止后续单元格运行。
    raise FileNotFoundError(f"未找到输入 WAV 文件：{wav_path}")
# 创建输出目录；父目录不存在时一并创建，目录已存在时不报错。
output_dir.mkdir(parents=True, exist_ok=True)

# 输出总体进度，说明输入检查和输出路径准备已经完成。
print("[######..........................]  20% 输入与输出路径准备完成")
# 输出实际使用的输入文件绝对路径，便于核对数据来源。
print(f"输入 WAV：{wav_path}")
# 输出实际使用的结果目录绝对路径，便于定位生成文件。
print(f"输出目录：{output_dir}")

[######..........................]  20% 输入与输出路径准备完成
输入 WAV：D:\Project_Github\Indo-Pacific-humpback-dolphin\00_Data\ClickTrains\PulseTrain_651\PulseTrain.wav
输出目录：D:\Project_Github\Indo-Pacific-humpback-dolphin\16_FpodStyleMultipath


## 第 3 步：读取 WAV 信号

将 WAV 文件读取为单通道信号，并输出采样率、样本数、时长和振幅范围。

In [3]:
# 读取 WAV 文件，并获得原始采样率和单通道时域信号数组。
sample_rate, signal = read_wav_mono(wav_path)
# 根据样本数量与采样率计算录音总时长，单位为秒。
duration_seconds = len(signal) / sample_rate

# 输出总体进度，说明音频数据已经成功载入内存。
print("[##########......................]  30% WAV 信号读取完成")
# 输出原始音频采样率，单位为赫兹。
print(f"原始采样率：{sample_rate} Hz")
# 输出单通道信号包含的采样点总数。
print(f"信号样本数：{len(signal):,}")
# 输出录音时长并保留六位小数，以便检查短脉冲序列。
print(f"录音时长：{duration_seconds:.6f} s")
# 输出原始信号的最小振幅。
print(f"最小振幅：{signal.min():.6f}")
# 输出原始信号的最大振幅。
print(f"最大振幅：{signal.max():.6f}")

[##########......................]  30% WAV 信号读取完成
原始采样率：576000 Hz
信号样本数：225,005
录音时长：0.390634 s
最小振幅：-0.685393
最大振幅：1.000000


## 第 4 步：执行带通滤波

使用 `DetectionSettings` 中的参数保留目标频带，抑制低频背景噪声和频带外成分。

In [4]:
# 创建点击检测配置对象，其中包含带通频率、阈值等检测参数。
detection_settings = DetectionSettings()
# 按检测配置对原始信号执行带通滤波，得到供后续点击检测使用的信号。
filtered = bandpass_filter(signal, sample_rate, detection_settings)

# 输出总体进度，说明目标频带信号已经提取完成。
print("[#############...................]  40% 带通滤波完成")
# 输出滤波器保留的低频截止值，单位为赫兹。
print(f"低频截止值：{detection_settings.lowcut_hz} Hz")
# 输出滤波器保留的高频截止值，单位为赫兹。
print(f"高频截止值：{detection_settings.highcut_hz} Hz")
# 输出滤波后信号的最小振幅，用于快速检查滤波结果的数值范围。
print(f"滤波后最小振幅：{filtered.min():.6f}")
# 输出滤波后信号的最大振幅，用于快速检查滤波结果的数值范围。
print(f"滤波后最大振幅：{filtered.max():.6f}")

[#############...................]  40% 带通滤波完成
低频截止值：20000.0 Hz
高频截止值：220000.0 Hz
滤波后最小振幅：-0.703037
滤波后最大振幅：0.955259


## 第 5 步：运行 F-POD 风格处理流程

从滤波信号中提取类似 FP1 的点击记录，并进一步组织类似 FP3 的点击串、多途簇与特征。

In [5]:
# 创建 F-POD 流程配置对象，用于控制重采样、记录筛选和聚类过程。
fpod_settings = FpodPdfSettings()
# 对滤波信号运行完整 F-POD 风格流程，并保存点击记录、特征和结构统计结果。
result = run_fpod_pdf_pipeline(filtered, sample_rate, fpod_settings)

# 检查流程是否成功提取类似 FP1 的点击记录。
if not result.click_records:
    # 没有点击记录时抛出异常，因为后续特征构建和绘图无法继续。
    raise RuntimeError("未能提取类似 FP1 的点击记录。")
# 检查流程是否根据点击记录成功构建多途特征。
if not result.features:
    # 没有多途特征时抛出异常，因为后续 CSV 导出和聚类绘图没有有效数据。
    raise RuntimeError("未能根据类似 FP1 的点击记录构建多途特征。")

# 输出总体进度，说明 F-POD 风格检测与特征构建已经完成。
print("[###################.............]  60% F-POD 风格处理流程完成")
# 输出流程内部使用的 F-POD 风格采样率，单位为赫兹。
print(f"F-POD 风格采样率：{result.sample_rate} Hz")
# 输出本次流程提取的底层点击记录总数。
print(f"类似 FP1 的点击记录数：{len(result.click_records):,}")
# 输出从点击记录构建得到的多途特征行数。
print(f"多途特征行数：{len(result.features):,}")
# 输出识别得到的点击串数量。
print(f"点击串数量：{result.train_count:,}")
# 输出识别得到的多途聚类数量。
print(f"多途聚类数量：{result.multipath_cluster_count:,}")
# 输出流程自动选择的检测参数，便于复核本次分析条件。
print(f"检测参数：{result.detection_choice}")

[###################.............]  60% F-POD 风格处理流程完成
F-POD 风格采样率：4000000 Hz
类似 FP1 的点击记录数：424
多途特征行数：424
点击串数量：10
多途聚类数量：52
检测参数：{'amplitude_percentile': 97.0, 'amplitude_threshold': 0.006768888463867043, 'click_records': 424.0, 'ipi_change_tolerance': 0.5}


## 第 6 步：导出点击记录和多途特征

分别保存底层点击记录和聚类特征，便于后续统计分析或在其他软件中检查。

In [ ]:
# 将类似 FP1 的底层点击记录写入指定 CSV 文件。
write_fpod_records_csv(result.click_records, records_csv_path)
# 将用于多途聚类和绘图的特征数据写入指定 CSV 文件。
write_features_csv(result.features, csv_path)

# 输出总体进度，说明两类结构化数据已经保存完成。
print("[#######################.........]  70% CSV 数据导出完成")
# 输出点击记录 CSV 的绝对路径及文件大小，便于确认文件确实写入磁盘。
print(f"点击记录 CSV：{records_csv_path}（{records_csv_path.stat().st_size:,} 字节）")
# 输出多途特征 CSV 的绝对路径及文件大小，便于确认文件确实写入磁盘。
print(f"多途特征 CSV：{csv_path}（{csv_path.stat().st_size:,} 字节）")

## 第 7 步：绘制完整时间范围的多途聚类图

首先输出覆盖全部点击时间的聚类图，并在当前 Notebook 中直接显示。

In [ ]:
# 使用全部多途特征绘制完整时间范围的聚类可视化图。
plot_multipath_cluster(result.features, png_path, title="PDF-structured F-POD click multipath cluster visualization")
# 输出总体进度，说明完整时间范围聚类图已经生成。
print("[##########################......]  80% 完整聚类图生成完成")
# 输出完整聚类图的保存路径，便于在文件系统中定位结果。
print(f"完整聚类图：{png_path}")
# 读取刚生成的 PNG，并将其嵌入当前 Notebook 输出区域。
display(Image(filename=str(png_path)))

## 第 8 步：绘制局部时间窗的多途聚类图

分别放大 80—120 毫秒和 30—130 毫秒时间窗，以观察局部点击与多途结构。

In [ ]:
# 绘制 80—120 毫秒时间窗内的多途聚类局部图。
plot_multipath_cluster(result.features, cropped_png_path, title="PDF-structured F-POD click multipath cluster visualization (80-120 ms)", time_start_ms=80.0, time_end_ms=120.0)
# 输出第一幅局部图的生成状态和保存路径。
print(f"80—120 ms 局部聚类图：{cropped_png_path}")
# 在当前 Notebook 中显示 80—120 毫秒局部聚类图。
display(Image(filename=str(cropped_png_path)))

# 绘制 30—130 毫秒时间窗内的多途聚类局部图。
plot_multipath_cluster(result.features, cropped_30_130_png_path, title="PDF-structured F-POD click multipath cluster visualization (30-130 ms)", time_start_ms=30.0, time_end_ms=130.0)
# 输出第二幅局部图的生成状态和保存路径。
print(f"30—130 ms 局部聚类图：{cropped_30_130_png_path}")
# 在当前 Notebook 中显示 30—130 毫秒局部聚类图。
display(Image(filename=str(cropped_30_130_png_path)))

# 输出总体进度，说明所有完整与局部聚类图均已生成。
print("[#############################...]  90% 所有聚类图生成完成")

## 第 9 步：汇总本次分析结果

集中输出采样率、检测参数、F-POD 结构数量和全部结果文件，作为本次运行的最终检查。

In [ ]:
# 输出完成状态和百分比，表示 Notebook 的全部分析步骤已经结束。
print("[################################] 100% 全部分析完成")
# 输出本次分析使用的输入 WAV 文件。
print(f"输入 WAV：{wav_path}")
# 输出输入音频的原始采样率。
print(f"原始采样率：{sample_rate} Hz")
# 输出 F-POD 风格流程内部使用的采样率。
print(f"F-POD 风格采样率：{result.sample_rate} Hz")
# 输出最终保留的类似 FP1 点击记录数量。
print(f"类似 FP1 的点击记录数：{int(result.detection_choice['click_records']):,}")
# 输出点击振幅筛选所采用的百分位数。
print(f"振幅百分位：p{result.detection_choice['amplitude_percentile']:.0f}")
# 输出脉冲间隔变化容差，用于复核点击串组织条件。
print(f"IPI 变化容差：{result.detection_choice['ipi_change_tolerance']:.2f}")
# 输出识别得到的点击串总数。
print(f"点击串数量：{result.train_count:,}")
# 输出识别得到的多途聚类总数。
print(f"多途聚类数量：{result.multipath_cluster_count:,}")
# 输出最终多途特征数据的行数。
print(f"特征行数：{len(result.features):,}")
# 输出类似 FP1 点击记录 CSV 的最终保存位置。
print(f"点击记录 CSV：{records_csv_path}")
# 输出多途特征 CSV 的最终保存位置。
print(f"多途特征 CSV：{csv_path}")
# 输出完整时间范围聚类图的最终保存位置。
print(f"完整聚类图：{png_path}")
# 输出 80—120 毫秒局部聚类图的最终保存位置。
print(f"80—120 ms 局部聚类图：{cropped_png_path}")
# 输出 30—130 毫秒局部聚类图的最终保存位置。
print(f"30—130 ms 局部聚类图：{cropped_30_130_png_path}")